# Lab 4: Swarm for Lambda error analysis

A user registration function on AWS Lambda is failing in production. In this lab a Strands `Swarm` gathers evidence about it. Three investigators each own one evidence source: the CloudWatch Logs, the source code, and the internal runbooks. After the Swarm stops, deterministic notebook code validates those tool results and invokes an analyzer exactly once to produce the root cause and fix. The analyzer also checks Amazon Bedrock AgentCore Memory for past incidents like this one.

No code in this lab says who works next. Each Agent reads what the team has found so far and hands the work to the teammate it thinks should look next. It does so with `handoff_to_agent`, a tool Strands adds to every Swarm member.

## What you will do

1. Run the evidence Swarm and watch each investigator in its own pane.
2. Validate its tool results, then run one final analyzer and inspect the memory records it used.
3. Run the investigator team with a tight limit and compare a limit-triggered stop with a completed evidence run.
4. Recall the incident the notebook persisted after validating the completed evidence packet, from a new session.

**Estimated time:** 20-25 minutes.

You do not edit the agent code. Run the cells from top to bottom. If you plan to complete optional Lab 6, run the clearly marked observability cell before the lab prepares its Memory data; otherwise skip that cell.

![Three investigators dynamically hand evidence to one another inside a Swarm. After deterministic validation, a separate analyzer runs exactly once, recalls past incidents from AgentCore Memory, and produces the root cause and fix.](images/topology.png)


## Why this is a Swarm

Nobody knows the investigation path before it starts. A traceback in the logs can point straight at a line of code, while an intermittent timeout with clean code needs the runbooks and past incidents too. The next specialist depends on what the current one found, so the Agent holding the work is the best judge of who looks next. That is the case a Swarm is built for. If you could write the route down in advance, a fixed route in code would be cheaper and easier to test.

Because no code fixes the route, a Swarm can also loop. `max_handoffs`, `max_iterations`, `execution_timeout`, `node_timeout`, and repetitive-handoff detection bound how long a run can take and how much it can cost. Limits do not fix unclear roles, though. If two Agents' descriptions overlap, the limit only decides where the ping-pong stops. Clear descriptions are what make the handoffs useful.

In Strands, a Swarm reports `COMPLETED` whenever the active Agent finishes without calling `handoff_to_agent`; that status alone does not prove the business investigation is complete. This lab therefore uses the Swarm only for dynamic evidence gathering. Deterministic Python checks that every required evidence tool actually ran before a separate analyzer is invoked once. A missing tool result fails clearly instead of being accepted as a completed investigation.

AgentCore Memory appears as the analyzer's `recall_past_incidents` tool. It is a retrieval source, like the logs are for the logs investigator. It is not how findings move between investigators: within the Swarm, findings travel in handoff messages and shared context. Across the deterministic boundary, the analyzer receives the validated tool outputs directly. After its evidence packet passes validation, notebook code persists the incident once. In the pinned `bedrock-agentcore` version, `AgentCoreMemorySessionManager` cannot hold a Swarm's state at all (its multi-agent methods raise `NotImplementedError`).

## Setup

Run the `%pip` cell by itself first. If it installs or upgrades anything, restart the kernel. Then run the import, model, and lab-resource cells in order.

When you reach **Optional: Enable observability**, run that cell if you plan to complete Lab 6; otherwise skip it. Then select the **Prepare the lab data** cell and choose **Run > Run All Below**. Starting **Run All** above the install cell can stop after a package-triggered kernel restart.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import json
import re
import sys
import time
import uuid
from pathlib import Path

import boto3
from bedrock_agentcore.memory import MemoryClient
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import Status, Swarm

sys.path.insert(0, str(Path("..").resolve()))
from utils.lab_data import EXTRACTION_WAIT_SECONDS, POLL_SECONDS, prepare_swarm_lab
from utils.live_panes import LivePanes, participant_text
from utils.stacks import cleanup, find_or_deploy
from utils.validation import write_validation_artifact

The investigators run on Anthropic Claude Haiku 4.5 and the analyzer on Anthropic Claude Sonnet 5, both in the AWS Region of this notebook's session.

In [ ]:
region = boto3.Session().region_name
haiku = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", region_name=region)
sonnet = BedrockModel(model_id="global.anthropic.claude-sonnet-5", region_name=region)

### Lab resources

The lab stack holds an AgentCore Memory with a semantic strategy and the Lambda function that fails on purpose. At an event the stack already exists. On your own account, the next cell deploys it, which takes a few minutes.

The optional observability cell comes next, before `prepare_swarm_lab()` writes or retrieves Memory data. Run it only if you plan to complete Lab 6.

In [ ]:
outputs = find_or_deploy("swarm", "infra/swarm.yaml")
print("Memory ID:", outputs["MemoryId"])

### Optional: Enable observability

Run the next cell if you plan to complete **Lab 6: Observability**. AgentCore Memory metrics are automatic; this cell enables Memory application logs and managed traces before Lab 4 creates, extracts, consolidates, and retrieves incident memories.

Continue only after the cell prints `Memory traces : enabled`. The configuration and its cleanup ownership are saved for Lab 6, so the Memory activity you generate in this lab remains available there. If you are not completing Lab 6, skip this cell.

In [ ]:
from utils.observability import enable_memory_observability

observability_state_path = Path("..") / ".lab6-observability-state.json"
lab6_observability = enable_memory_observability(
    outputs,
    observability_state_path,
    region=region,
)
assert lab6_observability["trace_delivery_ready"], (
    "Memory trace delivery is still activating. Wait a few minutes and rerun this cell before continuing."
)

### Prepare the lab data

`prepare_swarm_lab()` invokes the failing function so its log group holds a real traceback, writes four past incidents to Memory as events, and waits up to 180 seconds while the semantic strategy extracts them into long-term records. If extraction takes longer, the recall tool also returns the records in `data/fallback_records.json`, and the cell says `FIXTURE IN USE`.

In [ ]:
lab = prepare_swarm_lab(outputs, region)

## The tools

Each investigator gets one evidence tool, and the analyzer gets the memory recall tool. The source code and runbooks are small fixtures in `data/evidence.json`. The logs tool reads the real log group with the CloudWatch Logs `FilterLogEvents` API. `retrieve_memories` searches the long-term records of the on-call actor. After the Swarm finishes, notebook code calls `create_event` once, using the failing request ID as an idempotency key.

`evidence_calls` and `memory_calls` record every tool call, so the notebook can verify every required evidence owner and the exact runbook and past incident the analyzer used.

In [ ]:
# Evidence data, the CloudWatch Logs client, and the Memory client.
EVIDENCE = json.loads(Path("data/evidence.json").read_text())
logs = boto3.client("logs", region_name=region)
memory = MemoryClient(region_name=region)
# Each tool appends a record of its call here, so later cells can check what the Agents read.
evidence_calls = []
memory_calls = []


# Investigator tools: logs, source code, and runbooks.
# @tool turns a function into a tool: the docstring becomes its description, the type hints its input schema.
@tool
def read_cloudwatch_logs(service: str) -> str:
    """Return the last hour of CloudWatch Logs lines for a service's Lambda function."""
    if service == "user-registration-service":  # the function the lab stack deployed
        # The paginator follows nextToken across pages of filter_log_events results.
        pages = logs.get_paginator("filter_log_events").paginate(
            logGroupName=lab.log_group,
            startTime=int((time.time() - 3600) * 1000),  # one hour ago, in milliseconds since the epoch
            PaginationConfig={"MaxItems": 60},  # at most 60 events in total, across all pages
        )
        # search applies the JMESPath expression to every page and yields each message string.
        result = "".join(pages.search("events[].message")) or "No log events in the last hour."
    else:
        result = f"No logs found for {service}. Known services: {', '.join(EVIDENCE['services'])}."
    evidence_calls.append({
        "tool": "read_cloudwatch_logs",
        "service": service,
        "result": result,
    })
    return result


@tool
def read_source_code(service: str) -> str:
    """Return the handler source code of a service."""
    result = EVIDENCE["services"].get(service, {}).get("source", f"No source found for {service}.")
    evidence_calls.append({
        "tool": "read_source_code",
        "service": service,
        "result": result,
    })
    return result


@tool
def search_docs(query: str) -> str:
    """Search the internal runbooks for known error patterns and return the two best matches."""
    # A simple keyword search: rank runbooks by how many query words they contain.
    words = set(query.lower().split())
    ranked = sorted(EVIDENCE["docs"], key=lambda doc: -len(words & set(doc.lower().split())))
    result = "\n\n".join(ranked[:2])
    evidence_calls.append({
        "tool": "search_docs",
        "query": query,
        "result": result,
    })
    return result


# Analyzer tool: recall past incidents. Persistence happens once after validation.
@tool
def recall_past_incidents(query: str) -> str:
    """Search the records of past incidents for ones similar to the current symptom."""
    # A semantic search over long-term records in one namespace; it returns [] if the service call fails.
    records = memory.retrieve_memories(
        memory_id=lab.memory_id,
        namespace=lab.namespace,  # /incidents/{actorId}/ with the on-call actor ID filled in
        query=query,
        top_k=5,  # the five closest records
    )
    records += lab.fixture_records  # empty unless the setup cell printed FIXTURE IN USE
    memory_calls.append({
        "tool": "recall_past_incidents",
        "query": query,
        "records": [r["memoryRecordId"] for r in records],
        "texts": [r["content"]["text"] for r in records],
    })
    # Each record goes back to the Agent with its ID, so the answer can cite it.
    return "\n".join(
        f"[{r['memoryRecordId']}] {r['content']['text']}" for r in records
    ) or "No past incidents found."

## What to look for

The team starts at the logs investigator. The order after that is not fixed, but a complete investigation must include logs, source, runbook, and memory evidence before the analyzer can finish.

Look for:

- Which Agent does the logs investigator hand off to first?
- In what order do the code and docs investigators take their turns?
- Does the docs investigator find the **Null field handling** runbook?
- Which investigator finishes evidence gathering without another handoff?
- Does the separate analyzer recall `INC-0412`?
- How many investigator turns does the Swarm take?


## Run the swarm

The cell defines three investigator Agents and the evidence Swarm, then loops over `swarm.stream_async(task)`. Each event goes to `panes.show()`, which puts it in the pane of the investigator that produced it. The loop also keeps every `multiagent_handoff` event, because its `message` says why that handoff happened. The last event holds the `SwarmResult`.

Every Agent sets `callback_handler=None`: `stream_async` still calls each Agent's default handler, which would print the same text a second time below the panes.

**What to look for:** all three investigator panes, `[tool: handoff_to_agent]` while evidence is still missing, and `[tool: search_docs]` in the docs pane. The last investigator ends without a handoff once the shared context contains logs, source, and runbook findings. If an Agent receives work again, its pane shows `Turn 2`.

In [ ]:
# The investigation team. Descriptions are what teammates read when they choose whom to hand off to.
logs_investigator = Agent(
    name="logs_investigator",
    description="Reads a service's recent CloudWatch Logs and reports the exact error and when it happens.",
    model=haiku,
    system_prompt=(
        "You are a CloudWatch Logs investigator inside an evidence-gathering Swarm. Call "
        "read_cloudwatch_logs for the service named in the request and report the exact exception, the failing "
        "line, and any timing pattern. If source or runbook evidence is still missing from the shared context, "
        "your final action must be one handoff_to_agent call to the investigator who owns missing evidence; put "
        "your concise findings in its message. If logs, source, and runbook findings are all reported, finish with "
        "a concise evidence summary and do not hand off. "
        "Evidence counts as reported only after its owner reports its own tool result: logs from "
        "logs_investigator, source from code_investigator, runbook from docs_investigator. A source line "
        "quoted in a log traceback is not source evidence. A separate analyzer runs after validation."
    ),
    tools=[read_cloudwatch_logs],
    callback_handler=None,
)

code_investigator = Agent(
    name="code_investigator",
    description="Reads a service's source code and finds the line that explains an error, or says the code looks correct.",
    model=haiku,
    system_prompt=(
        "You are a source-code investigator inside an evidence-gathering Swarm. Call read_source_code for "
        "the service and point to the line that explains the reported error, or say plainly that the code looks "
        "correct. If logs or runbook evidence is still missing from the shared context, your final action must "
        "be one handoff_to_agent call to the investigator who owns missing evidence; put your concise findings "
        "in its message. If logs, source, and runbook findings are all reported, finish with a concise evidence "
        "summary and do not hand off. "
        "Evidence counts as reported only after its owner reports its own tool result: logs from "
        "logs_investigator, source from code_investigator, runbook from docs_investigator. A source line "
        "quoted in a log traceback is not source evidence. A separate analyzer runs after validation."
    ),
    tools=[read_source_code],
    callback_handler=None,
)

docs_investigator = Agent(
    name="docs_investigator",
    description="Matches the evidence so far against internal runbooks for known error patterns.",
    model=haiku,
    system_prompt=(
        "You are a runbook investigator inside an evidence-gathering Swarm. Call search_docs with the error and "
        "any pattern the other investigators found, and say which runbook fits. If logs or source evidence is "
        "still missing from the shared context, your final action must be one handoff_to_agent call to the "
        "investigator who owns missing evidence; put your concise findings in its message. If logs, source, and "
        "runbook findings are all reported, finish with a concise evidence summary and do not hand off. "
        "Evidence counts as reported only after its owner reports its own tool result: logs from "
        "logs_investigator, source from code_investigator, runbook from docs_investigator. A source line "
        "quoted in a log traceback is not source evidence. A separate analyzer runs after validation."
    ),
    tools=[search_docs],
    callback_handler=None,
)

# The swarm: entry point, limits, and repetitive-handoff detection.
# Swarm adds a handoff_to_agent tool to each Agent; calling it picks the next Agent and passes a message.
# The run ends when an Agent finishes its turn without calling it, or when a limit below stops the run.
swarm = Swarm(
    [logs_investigator, code_investigator, docs_investigator],
    entry_point=logs_investigator,  # the Agent that receives the task first
    max_handoffs=10,  # stop after 10 completed Agent turns
    max_iterations=12,  # also counts completed Agent turns; here max_handoffs stops the run first
    execution_timeout=600.0,  # seconds for the whole run
    node_timeout=180.0,  # seconds for any one Agent turn
    repetitive_handoff_detection_window=6,  # in any 6 consecutive turns ...
    repetitive_handoff_min_unique_agents=3,  # ... at least 3 different Agents must appear
)

# The incident and the evidence-gathering task. The analyzer later gets the incident text, not the task.
incident = (
    f"Lambda function user-registration-service is failing in production (latest failing request ID "
    f"{lab.request_id}) with AttributeError: 'NoneType' object has no attribute 'lower'."
)
task = (
    f"{incident} "
    "Gather CloudWatch logs, source code, and internal runbook evidence. Continue handing off while one of "
    "those evidence sources is missing; once all three are present, finish the evidence-gathering phase."
)

# Run the swarm, streaming each Agent and recording every handoff.
evidence_calls.clear()
memory_calls.clear()
handoffs = []
panes = LivePanes()
async for event in swarm.stream_async(task):
    panes.show(event)
    # A handoff event has from_node_ids, to_node_ids, and the message the sending Agent wrote.
    if event.get("type") == "multiagent_handoff":
        handoffs.append(event)
result = event["result"]  # the last event carries the SwarmResult

## Inspect the evidence

`node_history` is the path the investigators chose, and `execution_count` counts their turns. Each handoff message is the reason for one step of the path. The notebook then verifies the three evidence tool calls before invoking the analyzer exactly once. Its memory calls show whether it found a past incident, and which record it read. A record ID starting with `fixture-` comes from the fallback file, not from AgentCore Memory.

Compare this output with the questions under **What to look for**.

In [ ]:
# The run's status, path, and cost.
path = [node.node_id for node in result.node_history]  # node_history lists each completed Agent turn in order
print(f"status      : {result.status}")
print(f"path        : {' -> '.join(path)}")
print(f"agent turns : {result.execution_count}")
print(f"tokens      : {result.accumulated_usage['totalTokens']}")

# Preserve the complete exception before any validation assertion stops notebook execution.
# A failed node keeps its exception in node_result.result instead of an AgentResult.
failed_nodes = {
    name: node_result
    for name, node_result in result.results.items()
    if node_result.status == Status.FAILED
}
if failed_nodes:
    import traceback

    print("\nfailed node exceptions")
    for node_name, node_result in failed_nodes.items():
        print(f"  {node_name}: {type(node_result.result).__name__}: {node_result.result!r}")
        print("".join(traceback.format_exception(node_result.result)))

# Each handoff and the message it carried.
print("\nhandoffs, and why each one happened")
for handoff in handoffs:
    reason = " ".join(handoff["message"].split())  # collapse newlines and runs of spaces
    print(f"  {handoff['from_node_ids'][0]} -> {handoff['to_node_ids'][0]}: {reason[:160]}")

# Every evidence call made by the Swarm.
print("\nevidence tool calls")
for call in evidence_calls:
    print(f"  {call['tool']}: {' '.join(call['result'].split())[:120]}")

# Status.COMPLETED only means the last investigator did not hand off. Fail closed unless all evidence ran.
required_agents = {"logs_investigator", "code_investigator", "docs_investigator"}
required_tools = {"read_cloudwatch_logs", "read_source_code", "search_docs"}
observed_tools = {call["tool"] for call in evidence_calls}
missing_agents = required_agents - set(path)
missing_tools = required_tools - observed_tools
assert result.status == Status.COMPLETED, f"evidence Swarm status: {result.status}"
assert not missing_agents, (
    f"Swarm stopped without visiting required investigators: {sorted(missing_agents)}"
)
assert not missing_tools, (
    f"Swarm reported COMPLETED because its last Agent did not hand off, but evidence is missing: "
    f"{sorted(missing_tools)}"
)
assert any(
    "Runbook: Null field handling" in call["result"]
    for call in evidence_calls if call["tool"] == "search_docs"
), "the runbook evidence did not identify Null field handling"

# Cross a deterministic boundary: pass the validated raw tool outputs to one final analyzer invocation.
# The bundle holds what the tools returned, not the investigators' summaries of it.
evidence_bundle = "\n\n".join(
    f"{call['tool']} result:\n{call['result']}" for call in evidence_calls
)
analyzer = Agent(
    name="analyzer",
    model=sonnet,
    system_prompt=(
        "You are the final incident analyzer, outside the Swarm. First call recall_past_incidents with the "
        "incident symptom. Then synthesize only the validated logs, source, and runbook tool outputs supplied "
        "in the request plus the memory records returned by your tool. Answer with explicit Logs:, Source:, "
        "Runbook:, and Memory: lines, then Root cause:, Fix:, and Confidence:. Name the matching runbook and "
        "past incident ID."
    ),
    tools=[recall_past_incidents],
    callback_handler=None,
)
memory_calls.clear()
panes = LivePanes()
async for event in analyzer.stream_async(
    f"Incident:\n{incident}\n\nValidated evidence tool outputs:\n{evidence_bundle}"
):
    panes.show(event)

print("\nmemory tool calls")
for call in memory_calls:
    print(f"  {call['tool']}: {call['query']}")
    for record_id in call["records"]:
        print(f"      read {record_id}")

# The analyzer's participant-facing evidence packet.
required_labels = ("Logs", "Source", "Runbook", "Memory", "Root cause", "Fix", "Confidence")


def has_label(text, label):
    """True for "Label:" anywhere, or for a line that starts with the label as plain text, bold, or a heading."""
    # "Label:" or "**Label**:", as a whole word, with optional spaces before the colon.
    with_colon = rf"\b{label}\b\**\s*:"
    # Heading marks, bold stars, list dashes, or spaces, then the label at the start of a line.
    at_line_start = rf"^[#*\s-]*{label}\b"
    # MULTILINE lets ^ match at the start of every line, not only the start of the text.
    return re.search(f"{with_colon}|{at_line_start}", text, flags=re.IGNORECASE | re.MULTILINE) is not None


# One string per assistant message: join its text blocks (skipping tool calls), then strip reasoning blocks.
analyzer_messages = [
    participant_text("".join(block["text"] for block in message["content"] if "text" in block)).strip()
    for message in analyzer.messages if message["role"] == "assistant"
]
complete_packets = [
    text
    for text in analyzer_messages
    if all(has_label(text, label) for label in required_labels)
]
assert complete_packets, "the analyzer did not produce a complete evidence packet"
analyzer_answer = complete_packets[-1]
print(f"\nanalyzer evidence packet\n{analyzer_answer}")

# Validate the deterministic analyzer result before persistence.
assert any("INC-0412" in text for call in memory_calls for text in call["texts"]), "INC-0412 was not recalled"
for label in required_labels:
    assert has_label(analyzer_answer, label), f"the analyzer omitted {label}"
assert "Null field handling" in analyzer_answer, "the final answer omitted the runbook"
assert "INC-0412" in analyzer_answer, "the final answer omitted the past incident"

# Persist exactly once after validation. Rerunning this cell reuses the same request-scoped session.
incident_id = f"user-registration-service:{lab.request_id}"
incident_session_id = f"incident-{lab.request_id}"  # one session per failing request, so a rerun finds it
# The long-term record IDs that exist now; the next cells look for records that are new after this point.
# MemoryClient passes list_memory_records to the boto3 API, so it returns the raw response, one page.
records_before_persist = {
    record["memoryRecordId"]
    for record in memory.list_memory_records(
        memoryId=lab.memory_id,
        namespace=lab.namespace,
        maxResults=100,
    )["memoryRecordSummaries"]
}
# Short-term events are stored per actor and session. Two results tell none, one, or a duplicate apart.
existing = memory.list_events(
    memory_id=lab.memory_id,
    actor_id=lab.actor_id,  # the on-call team, the same actor as the seeded past incidents
    session_id=incident_session_id,
    max_results=2,
)
if existing:
    persisted_event, persistence_created = existing[0], False
else:
    # create_event stores a short-term event; the semantic strategy extracts long-term records later.
    persisted_event = memory.create_event(
        memory_id=lab.memory_id,
        actor_id=lab.actor_id,
        session_id=incident_session_id,
        # The recurrence line states what is new, so consolidation has a fact to add, not only a repeat.
        messages=[
            (
                f"Incident ID: {incident_id}\n"
                f"Recurrence of INC-0412: request {lab.request_id} failed with the same error, "
                "so the INC-0412 fix is not in the deployed code.\n"
                f"{analyzer_answer}",
                "USER",
            ),
        ],  # (text, role) pairs
        metadata={
            "incidentId": {
                "stringValue": incident_id,
            },
        },
    )
    persistence_created = True
persisted = memory.list_events(
    memory_id=lab.memory_id,
    actor_id=lab.actor_id,
    session_id=incident_session_id,
    max_results=2,
)
assert len(persisted) == 1, f"expected one event for {incident_id}, found {len(persisted)}"
print(f"incident persistence: {'created' if persistence_created else 'already existed'} {persisted_event['eventId']}")

## A limit-triggered stop

A limit ends a run with `status` `FAILED` and no exception, and it can stop the team one step short of another investigator. In this Strands version `max_handoffs` and `max_iterations` both count completed Agent turns. The check runs before each turn, so a handoff can be accepted and then never run.

The next cell reruns the same team with `max_handoffs=1`. Only the entry point, the logs investigator, takes its turn. It can still hand off, but the limit stops the run before the next Agent starts.

Look for:

- What is `status`?
- Which Agent received the last handoff, and did it run?
- Which panes appear, and in which state does each end?

In [ ]:
max_handoffs_1 = 1  # one completed turn, then the limit stops the run

# A fresh team: a Swarm adds handoff_to_agent to its Agents, so each Swarm needs new Agents.
logs_investigator = Agent(
    name="logs_investigator",
    description="Reads a service's recent CloudWatch Logs and reports the exact error and when it happens.",
    model=haiku,
    system_prompt=(
        "You are a CloudWatch Logs investigator inside an evidence-gathering Swarm. Call "
        "read_cloudwatch_logs for the service named in the request and report the exact exception, the failing "
        "line, and any timing pattern. If source or runbook evidence is still missing from the shared context, "
        "your final action must be one handoff_to_agent call to the investigator who owns missing evidence; put "
        "your concise findings in its message. If logs, source, and runbook findings are all reported, finish with "
        "a concise evidence summary and do not hand off. "
        "Evidence counts as reported only after its owner reports its own tool result: logs from "
        "logs_investigator, source from code_investigator, runbook from docs_investigator. A source line "
        "quoted in a log traceback is not source evidence. A separate analyzer runs after validation."
    ),
    tools=[read_cloudwatch_logs],
    callback_handler=None,
)

code_investigator = Agent(
    name="code_investigator",
    description="Reads a service's source code and finds the line that explains an error, or says the code looks correct.",
    model=haiku,
    system_prompt=(
        "You are a source-code investigator inside an evidence-gathering Swarm. Call read_source_code for "
        "the service and point to the line that explains the reported error, or say plainly that the code looks "
        "correct. If logs or runbook evidence is still missing from the shared context, your final action must "
        "be one handoff_to_agent call to the investigator who owns missing evidence; put your concise findings "
        "in its message. If logs, source, and runbook findings are all reported, finish with a concise evidence "
        "summary and do not hand off. "
        "Evidence counts as reported only after its owner reports its own tool result: logs from "
        "logs_investigator, source from code_investigator, runbook from docs_investigator. A source line "
        "quoted in a log traceback is not source evidence. A separate analyzer runs after validation."
    ),
    tools=[read_source_code],
    callback_handler=None,
)

docs_investigator = Agent(
    name="docs_investigator",
    description="Matches the evidence so far against internal runbooks for known error patterns.",
    model=haiku,
    system_prompt=(
        "You are a runbook investigator inside an evidence-gathering Swarm. Call search_docs with the error and "
        "any pattern the other investigators found, and say which runbook fits. If logs or source evidence is "
        "still missing from the shared context, your final action must be one handoff_to_agent call to the "
        "investigator who owns missing evidence; put your concise findings in its message. If logs, source, and "
        "runbook findings are all reported, finish with a concise evidence summary and do not hand off. "
        "Evidence counts as reported only after its owner reports its own tool result: logs from "
        "logs_investigator, source from code_investigator, runbook from docs_investigator. A source line "
        "quoted in a log traceback is not source evidence. A separate analyzer runs after validation."
    ),
    tools=[search_docs],
    callback_handler=None,
)

# The same swarm, now stopped by max_handoffs_1.
swarm = Swarm(
    [logs_investigator, code_investigator, docs_investigator],
    entry_point=logs_investigator,
    max_handoffs=max_handoffs_1,
    max_iterations=12,
    execution_timeout=600.0,
    node_timeout=180.0,
    repetitive_handoff_detection_window=6,
    repetitive_handoff_min_unique_agents=3,
)

# Run until the limit stops the swarm.
evidence_calls.clear()
memory_calls.clear()
limited_handoffs = []
panes = LivePanes()
async for event in swarm.stream_async(task):
    panes.show(event)
    if event.get("type") == "multiagent_handoff":
        limited_handoffs.append(event)
limited_result = event["result"]  # the last event carries the SwarmResult, even when a limit stopped the run

### Verify

Compare this with the first run. The panes already show it: the Agent that received the last handoff finished as `Stopped`, because the limit ended the run before its turn.

A limit caps how long a run takes and what it costs. It says nothing about the investigation, and here it cut off a healthy one. Repetitive-handoff detection stops a different failure. Give two Agents overlapping descriptions and they can trade the work back and forth. The run then stops once fewer than `repetitive_handoff_min_unique_agents` distinct Agents appear in the window.

In [ ]:
limited_path = [node.node_id for node in limited_result.node_history]
print(f"status       : {limited_result.status}")
print(f"path         : {' -> '.join(limited_path)}")
print(f"memory calls : {[call['tool'] for call in memory_calls]}")

# A Swarm that stops on a limit reports FAILED, not COMPLETED, and keeps the turns it finished.
assert limited_result.status == Status.FAILED, (
    f"status: {limited_result.status}; expected the one-turn limit to stop the evidence Swarm"
)
last = limited_handoffs[-1]
print(f"last handoff : {last['from_node_ids'][0]} -> {last['to_node_ids'][0]}")
assert limited_result.execution_count == max_handoffs_1, "the limit counts completed Agent turns"

## Recall in a new session

After the first run's evidence packet passed validation, deterministic notebook code wrote one request-scoped event to memory. The semantic strategy extracts it into a long-term record in the on-call actor's namespace, `/incidents/lambda-oncall/`. Records in that namespace are visible from any session, so the next investigation can find them without any conversation history.

Look for:

- Does the swarm's incident come back as a new long-term record?
- With no conversation history, does a new Agent find the record the swarm wrote?

The semantic strategy may summarize the event without copying the request ID verbatim. The next cell therefore identifies records created from this event by their creation time and the actor's namespace, rather than searching generated summary text for the ID. Extraction usually finishes within a minute or two.

Extraction is followed by consolidation. For each extracted memory, the strategy compares it with similar records in the namespace and chooses to add a record, update one, or do nothing because the memory is redundant. The event states that INC-0412 happened again, which is new information, but the strategy can still judge it redundant. In that case no new record appears, the cell says so, and the next check looks for INC-0412 instead.

In [ ]:
# Extraction runs in the background, so poll for new long-term records until the deadline.
deadline = time.monotonic() + EXTRACTION_WAIT_SECONDS  # the same wait as the setup cell
event_time = persisted_event["eventTimestamp"]
while True:
    # MemoryClient passes list_memory_records to the boto3 API, so it returns the raw response, one page.
    records = memory.list_memory_records(
        memoryId=lab.memory_id,
        namespace=lab.namespace,
        maxResults=100,
    )
    # Match by time and ID, not by text: extraction may rewrite the packet's wording.
    written = [
        record
        for record in records["memoryRecordSummaries"]
        if record["createdAt"] >= event_time
        and record["memoryRecordId"] not in records_before_persist
    ]
    # On a rerun, the earlier run's records are already in records_before_persist, so match by time only.
    if not persistence_created:
        written = [
            record
            for record in records["memoryRecordSummaries"]
            if record["createdAt"] >= event_time
        ]
    if written or time.monotonic() > deadline:
        break
    time.sleep(POLL_SECONDS)

for record in written:
    print(f"{record['memoryRecordId']}  created {record['createdAt']:%H:%M:%S}")
    print(f"  {' '.join(record['content']['text'].split())[:200]}")
# Consolidation compares each extracted memory with similar records and picks ADD, UPDATE, or NO-OP.
# NO-OP writes no record, so an empty result here is an outcome to report, not an error.
if not written:
    print(
        f"No new long-term record from event {persisted_event['eventId']} within {EXTRACTION_WAIT_SECONDS} seconds. "
        "The semantic strategy most likely judged it redundant with INC-0412 (NO-OP), "
        "or extraction is still running. The next cell recalls the existing record."
    )

A new on-call Agent in a new session asks about the first incident. It has never seen the swarm's conversation. The check reads the records its recall returned and confirms that one of them is the record the swarm wrote, or INC-0412 if consolidation wrote no new record, whatever the Agent's answer says.

In [ ]:
# A new on-call Agent in a new session, with only the recall tool.
session_id = str(uuid.uuid4())  # a new session: nothing from the swarm's conversation carries over
oncall = Agent(
    name="oncall",
    model=sonnet,
    system_prompt="You answer on-call questions. Call recall_past_incidents first, then answer in at most three sentences.",
    tools=[recall_past_incidents],
    callback_handler=None,
)

# Ask about the incident the swarm recorded.
memory_calls.clear()
panes = LivePanes()
async for event in oncall.stream_async(
    "Has user-registration-service failed on a missing email before? What fixed it?"
):
    panes.show(event)


# Check that the recall returned the swarm's record, or INC-0412 if consolidation wrote none.
recalled = {record_id for call in memory_calls for record_id in call["records"]}
# The intersection keeps the recalled IDs that the polling cell found as new.
from_swarm = recalled & {r["memoryRecordId"] for r in written}
print(f"records recalled   : {sorted(recalled)}")
print(f"written by the swarm: {sorted(from_swarm)}")
if written:
    assert from_swarm, "the recall did not return a record the swarm wrote"
else:
    assert any("INC-0412" in text for call in memory_calls for text in call["texts"]), "the recall did not return INC-0412"

## Clean up

If you enabled observability for Lab 6, the next cell keeps the Lab 4 stack so Lab 6 can inspect its Memory activity. Lab 6 removes the workshop-owned telemetry configuration and deletes a self-paced fallback stack in its final cleanup cell.

Otherwise, the cell deletes the stack only if this notebook deployed it. Workshop Studio event stacks are always left in place.

In [ ]:
observability_state_path = Path("..") / ".lab6-observability-state.json"
if observability_state_path.exists():
    print("Keeping the Lab 4 Memory and its telemetry for optional Lab 6.")
else:
    cleanup("swarm")

In [ ]:
# A completion marker in validation/swarm.json, so a facilitator can verify the run without saved output.
write_validation_artifact("swarm", {
    "status": result.status.value,
    "path": path,
    "required_agents": sorted(required_agents),
    "required_tools": sorted(required_tools),
    "final_analyzer_invocations": 1,
    "incident_id": incident_id,
    "event_id": persisted_event["eventId"],
    "new_record": bool(written),
    "recalled_records": sorted(from_swarm),
})

## Takeaways

In a Swarm, the Agent holding the work decides who takes it next. The code declares the investigator team, the entry point, and the limits, and nothing about the evidence route. Descriptions are the contract between teammates, and the handoff message records why each step happened.

A Swarm's `COMPLETED` status means its active Agent ended without another handoff; it does not prove that required business evidence exists. This lab therefore checks the investigator path and tool-call ledger before crossing a deterministic boundary to one final analyzer invocation. There is no whole-Swarm retry and no retry loop.

A limit ends a run as `FAILED` without raising, and `node_history` shows how far the team got. Both limit failures and no-handoff completion need application-level validation before their results are used.

AgentCore Memory served here as the final analyzer's retrieval tool, with its own owner (the on-call actor) and lifetime (the memory's event expiry and the stack). It carried knowledge between sessions, while handoffs carried findings within the evidence Swarm.

Amazon Bedrock charged for the model calls, and AgentCore for the stored events, extraction, and retrieval. Every handoff is another model call.

## Where to go next

The [Swarm guide](https://strandsagents.com/docs/user-guide/sdk/multi-agent/swarm/) covers shared `invocation_state`, session persistence with the file and S3 session managers, and interrupts. The [AgentCore Memory guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html) covers the other long-term strategies and custom extraction.

## Source and license

The lab uses the core `strands.multiagent.Swarm` class, and its limit settings were informed by the [Strands Agents Swarm sample](https://github.com/strands-agents/samples/tree/main/python/01-learn/11-swarm), licensed under Apache License 2.0. See `LICENSE` and `NOTICE`.